# Synthetic Random Matrix Benchmark: Pre-Norm vs GOR vs mHC (2026) vs AttnRes (2026) vs Gated Residual (2026)
Notebook này chạy trực tiếp trên **GPU Colab (A100 / L4 / T4)** để đo đạc khách quan 5 kiến trúc dòng dư trên bộ dữ liệu ma trận số ngẫu nhiên đa tầng (`8 → 128` tầng) và đa quy mô dữ liệu (`2.048 → 16.384` mẫu):
1. **Controlled Cosine-Alignment Stress Test (`L = 64, 128`):** Kiểm tra lan truyền xuôi/ngược và tỷ lệ xung đột âm.
2. **Depth Scaling (`L = 8, 16, 32, 64, 128`):** Đo tăng trưởng chuẩn $\|x_l\|_2$, Effective Rank, Gradient Stability, và tốc độ thực tế.
3. **Multi-Seed Training Benchmarks (Markov, Structured Teacher, Outlier Injection):** So sánh Loss hội tụ thực tế của 5 kiến trúc.
4. **Multi-Depth × Multi-Data Scaling Grid (`L = 12, 32` × `N = 2048, 16384`):** Kiểm chứng thực sự **GOR** mạnh ở đâu và yếu ở đâu so với **mHC**, **AttnRes**, **Gated Residual**.

## 1. Kiểm Tra GPU & Cài Thư Viện

In [ ]:
import os
os.system("nvidia-smi")
os.system("pip install -q torch matplotlib numpy")
import torch
print("PyTorch:", torch.__version__, "| CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Tải Mã Nguồn `residual_benchmark` Từ GitHub

In [ ]:
import os
if not os.path.exists("Alba-model"):
    os.system("git clone https://github.com/potatips/Alba-model.git")
else:
    os.system("cd Alba-model && git pull origin main")
os.chdir("Alba-model")
print("Working Directory:", os.getcwd())

## 3. Chạy Unit Test Kiểm Chứng Công Thức Cả 5 Kiến Trúc (14 Tests)

In [ ]:
import subprocess
res = subprocess.run(["python3", "-m", "unittest", "residual_benchmark.test_residual_benchmark"], capture_output=True, text=True)
print(res.stdout)
print(res.stderr)
assert res.returncode == 0, "Unit tests failed!"

## 4. Chạy Full Benchmark Trên GPU CUDA (Nhiều Tầng & Nhiều Dữ Liệu)
Bạn có thể tăng `--dim 256` hoặc `--train-layers 48` hay `--train-samples 32768` tùy thích trên A100.

In [ ]:
import subprocess
import torch

dev = "cuda" if torch.cuda.is_available() else "cpu"
cmd = [
    "python3", "-m", "residual_benchmark.run_benchmark",
    "--device", dev,
    "--dim", "256",
    "--seq-len", "16",
    "--train-layers", "32",
    "--train-samples", "16384",
    "--train-steps", "240",
    "--batch-size", "128",
    "--num-seeds", "2",
    "--output-dir", "residual_benchmark/results_colab",
]
print("Executing:", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
assert proc.returncode == 0, f"Benchmark exited with code {proc.returncode}"

## 5. Xem Biểu Đồ, Báo Cáo Markdown & Tải Kết Quả Về Máy

In [ ]:
from IPython.display import Image, Markdown, display
import os

out_dir = "residual_benchmark/results_colab"
png_path = os.path.join(out_dir, "residual_benchmark_2026.png")
md_path = os.path.join(out_dir, "BENCHMARK_REPORT.md")
json_path = os.path.join(out_dir, "benchmark_results.json")

if os.path.exists(png_path):
    display(Image(png_path))
if os.path.exists(md_path):
    with open(md_path, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))

try:
    from google.colab import files
    files.download(md_path)
    files.download(png_path)
    files.download(json_path)
except Exception:
    print("Saved outputs in:", out_dir)